# lmm_neural_efficiency.ipynb\n\n**Purpose:** Fit linear mixed-effects models (LMM) to test whether the\nNeural Efficiency (NE) index differs in its longitudinal trajectory between\naccelerated and non-accelerated reading groups.\n\n**Inputs:**\n- nirs_neural_efficiency.xlsx — output of nirs_neural_efficiency.ipynb\n\n**Outputs:**\n- lmm_results_by_roi.xlsx — LMM coefficients, SE, z, p, CI for frontal and temporal ROIs\n- Forest and trajectory plots (PNG, 300 dpi)\n\n**Model:** NE ~ group * sessao_num + VELmed_baseline + (1|subject) [REML]\n- Two separate models: one per ROI (FRONTAL, TEMPORAL)\n- Reference group: 
ao_acelerado; N = 126 obs per ROI (14 × 9 sessions)\n\n**Library:** statsmodels 0.14.6 (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# LMM — Neural Efficiency
**Project SESI | Input: fnirs_neural_efficiency.xlsx**

Two separate models by ROI (Frontal / Temporal)  
Outcome: `neural_efficiency`  
Fixed effects: `group * sessao_num + VELmed_baseline`  
Random effect: `(1 | subject)`

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration

In [ ]:
# Update this path to match your local data directory
PATH_IN  = r'../data/fnirs_neural_efficiency.xlsx'
# Update this path to match your local data directory
PATH_OUT_EXCEL = r'../results/lmm_results_by_roi.xlsx'
# Update this path to match your local data directory
PATH_OUT_FIG   = r'../results/lmm_results_by_roi.png'

FORMULA = "neural_efficiency ~ group * sessao_num + VELmed_baseline"
REF_GROUP = 'nao_acelerado'  # reference category

## 2 — Load and prepare

In [ ]:
df = pd.read_excel(PATH_IN)

# Set reference category for group
df['group'] = pd.Categorical(
    df['group'],
    categories=[REF_GROUP, 'acelerado'],
    ordered=False
)
df['subject'] = pd.Categorical(df['subject'])

# Sanity checks
print(f'Shape          : {df.shape}')
print(f'Subjects       : {df["subject"].nunique()}  (expected 14)')
print(f'Sessions       : {sorted(df["sessao_num"].unique())}')
print(f'ROIs           : {df["roi"].unique().tolist()}')
print(f'Missing NE     : {df["neural_efficiency"].isna().sum()}')
print(f'Missing VELbase: {df["VELmed_baseline"].isna().sum()}')

# Quick correlation: VELmed_baseline vs neural_efficiency
r = df[['VELmed_baseline','neural_efficiency']].corr().iloc[0,1]
print(f'\nCorr(VELmed_baseline, NE): {r:.3f}')
print('(Negative expected if faster baseline → lower NE — check for artefact)')

# Split by ROI
df_frontal  = df[df['roi'] == 'FRONTAL'].copy()
df_temporal = df[df['roi'] == 'TEMPORAL'].copy()
print(f'\nFrontal rows : {len(df_frontal)}')
print(f'Temporal rows: {len(df_temporal)}')

## 3 — Helper functions

In [ ]:
PARAM_LABELS = {
    f'group[T.acelerado]'            : 'Group Effect (Accelerated)',
    'sessao_num'                     : 'Session Progression',
    'group[T.acelerado]:sessao_num'  : 'Training-Induced NE Gain',
    'VELmed_baseline'                : 'Baseline Reading Time',
}


def lmm_to_df(res):
    """Convert LMM result to a clean DataFrame — fixed effects only."""
    fe_index = res.fe_params.index          # only fixed effects
    ci       = res.conf_int().loc[fe_index]
    pvals    = res.pvalues.loc[fe_index]
    bse      = res.bse.loc[fe_index]

    return pd.DataFrame({
        'parameter' : fe_index,
        'coef'      : res.fe_params.values,
        'se'        : bse.values,
        'z'         : (res.fe_params / bse).values,
        'pvalue'    : pvals.values,
        'ci_lower'  : ci[0].values,
        'ci_upper'  : ci[1].values,
    }).assign(significant=lambda d: d['pvalue'] < 0.05)


def forest_plot(res, title, color, path):
    """Forest plot — significant params colored, others grey."""
    df_res = lmm_to_df(res)
    df_res = df_res[df_res['parameter'] != 'Intercept'].copy()
    df_res['label'] = df_res['parameter'].map(PARAM_LABELS).fillna(df_res['parameter'])
    df_res = df_res.iloc[::-1].reset_index(drop=True)

    colors = [color if s else 'lightgrey' for s in df_res['significant']]

    fig, ax = plt.subplots(figsize=(12, 5))
    for i, row in df_res.iterrows():
        c = colors[i]
        ax.errorbar(
            y=i, x=row['coef'],
            xerr=[[row['coef'] - row['ci_lower']], [row['ci_upper'] - row['coef']]],
            fmt='o', color=c, capsize=5,
            markersize=9, markeredgecolor='black',
            ecolor=c, linewidth=1.5
        )
        # p-value annotation
        p_label = f"p={row['pvalue']:.3f}" if row['pvalue'] >= 0.001 else 'p<0.001'
        ax.text(row['ci_upper'] + 0.02, i, p_label, va='center', fontsize=9,
                color='black' if row['significant'] else 'grey')

    ax.set_yticks(range(len(df_res)))
    ax.set_yticklabels(df_res['label'], fontsize=11)
    ax.axvline(0, linestyle='--', color='grey', linewidth=1.2)

    sig_patch   = mpatches.Patch(color=color,      label='p < 0.05')
    insig_patch = mpatches.Patch(color='lightgrey', label='p ≥ 0.05')
    ax.legend(handles=[sig_patch, insig_patch], fontsize=10)

    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Coefficient (β)', fontsize=11)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    print(f'  -> Figure saved: {path}')
    plt.show()
    plt.close()

## 4 — Model A: Frontal ROI

In [ ]:
print('=' * 60)
print('MODEL A — FRONTAL ROI')
print('=' * 60)
print(f'N obs   : {len(df_frontal)}')
print(f'N groups: {df_frontal["subject"].nunique()}')

lmm_a = smf.mixedlm(FORMULA, data=df_frontal, groups=df_frontal['subject'])
res_a = lmm_a.fit(reml=True)
print(res_a.summary())

forest_plot(
    res_a,
    title = 'Neural Efficiency — Frontal ROI',
    color = 'steelblue',
    path  = PATH_OUT_FIG + 'forest_frontal.png'
)

## 5 — Model B: Temporal ROI

In [ ]:
print('=' * 60)
print('MODEL B — TEMPORAL ROI')
print('=' * 60)
print(f'N obs   : {len(df_temporal)}')
print(f'N groups: {df_temporal["subject"].nunique()}')

lmm_b = smf.mixedlm(FORMULA, data=df_temporal, groups=df_temporal['subject'])
res_b = lmm_b.fit(reml=True)
print(res_b.summary())

forest_plot(
    res_b,
    title = 'Neural Efficiency — Temporal ROI',
    color = 'darkorange',
    path  = PATH_OUT_FIG + 'forest_temporal.png'
)

## 6 — Side-by-side comparison

In [ ]:
df_a = lmm_to_df(res_a).assign(roi='FRONTAL')
df_b = lmm_to_df(res_b).assign(roi='TEMPORAL')
df_comp = pd.concat([df_a, df_b])
df_comp['label'] = df_comp['parameter'].map(PARAM_LABELS).fillna(df_comp['parameter'])

# Print key interaction term for both models
key = 'group[T.acelerado]:sessao_num'
print('Key result — Training-Induced NE Gain (group × session):')
print(df_comp[df_comp['parameter'] == key][['roi','coef','pvalue','ci_lower','ci_upper','significant']].to_string(index=False))

## 7 — Trajectory plot (Frontal and Temporal side by side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

roi_config = [
    ('FRONTAL',  df_frontal,  axes[0], 'Frontal ROI',  'steelblue',  'royalblue'),
    ('TEMPORAL', df_temporal, axes[1], 'Temporal ROI', 'darkorange', 'chocolate'),
]

for roi, data, ax, subtitle, c_acc, c_ctrl in roi_config:
    traj = (
        data.groupby(['sessao_num', 'group'], observed=True)['neural_efficiency']
        .agg(['mean', 'sem'])
        .reset_index()
    )

    for grp, color, label in [
        ('acelerado',    c_acc,  'Accelerated'),
        ('nao_acelerado', c_ctrl, 'Non-Accelerated')
    ]:
        sub = traj[traj['group'] == grp].sort_values('sessao_num')
        ax.plot(sub['sessao_num'], sub['mean'],
                marker='o', color=color, label=label, linewidth=2)
        ax.fill_between(
            sub['sessao_num'],
            sub['mean'] - sub['sem'],
            sub['mean'] + sub['sem'],
            alpha=0.2, color=color
        )

    ax.axhline(0, linestyle='--', color='grey', linewidth=1)
    ax.set_title(subtitle, fontsize=13, fontweight='bold')
    ax.set_xlabel('Session', fontsize=11)
    ax.set_ylabel('Neural Efficiency (mean ± SEM)', fontsize=11)
    ax.set_xticks(range(1, 10))
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(fontsize=10)

fig.suptitle('Neural Efficiency Trajectory by Group and ROI', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
traj_path = PATH_OUT_FIG + 'trajectory_by_roi.png'
plt.savefig(traj_path, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {traj_path}')
plt.show()

## 8 — Export results to Excel

In [ ]:
with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    lmm_to_df(res_a).to_excel(writer, sheet_name='LMM_FRONTAL',  index=False)
    lmm_to_df(res_b).to_excel(writer, sheet_name='LMM_TEMPORAL', index=False)
    df_comp.to_excel(writer,           sheet_name='COMPARISON',   index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print('   Sheets: LMM_FRONTAL | LMM_TEMPORAL | COMPARISON')